In [ ]:
import json
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool
from scipy.stats import zscore
from sklearn.cluster import KMeans

In [ ]:
# --- Charger la version agrégée "1 ligne = 1 devis" ---
import pandas as pd
from pathlib import Path

DATA = next((p for p in ["../data/quotes_grouped.json","../data/quotes_grouped.csv"] if Path(p).exists()), None)
assert DATA is not None, "Fichier agrégé introuvable (quotes_grouped.json/csv)."

df = pd.read_json(DATA) if DATA.endswith(".json") else pd.read_csv(DATA)

# Colonnes attendues : source_language, target_language, unit,
# number_of_unit, document_type, requested_deadline, proposed_price_accepted
df = df[df["proposed_price_accepted"].notnull()].copy()

# Feature engineering identique
df["requested_deadline"] = pd.to_numeric(df["requested_deadline"], errors="coerce")
df["requested_deadline"] = df["requested_deadline"].fillna(df["requested_deadline"].median())
df["is_express"] = (df["requested_deadline"] <= 2).astype(int)
df["flexibility"] = df["requested_deadline"] - 1
df["lang_pair"] = df["source_language"].astype(str) + "_to_" + df["target_language"].astype(str)
df["doc_category"] = df["document_type"].str.split("|").str[0]
df["doc_subcategory"] = df["document_type"].str.split("|").str[1]

# On enlève document_type (on garde les 2 découpages)
df.drop(columns=["document_type"], inplace=True)

# IQR + clustering (tes cellules actuelles) : tu peux les laisser identiques
# puis on continue comme dans ton notebook (X, y, train_test_split, Optuna, SHAP, etc.)

In [ ]:
# === Sanity-clean des features avant CatBoost ===
import numpy as np, pandas as pd

TARGET = "proposed_price_accepted"

# 1) Retirer les identifiants / colonnes non-modélisables si présents
ID_COLS = [c for c in ["quote_no", "id", "created_at"] if c in df.columns]
df_model = df.drop(columns=ID_COLS, errors="ignore").copy()

# 2) X, y (y en log pour rester cohérent)
y = np.log1p(df_model[TARGET].astype(float))
X = df_model.drop(columns=[TARGET])

# 3) Caster explicitement les catégorielles
CATS_CANON = ["source_language", "target_language", "unit",
              "lang_pair", "doc_category", "doc_subcategory"]
for c in CATS_CANON:
    if c in X.columns:
        X[c] = X[c].astype("category")

# 4) Liste réelle des colonnes catégorielles
cat_cols = [c for c in CATS_CANON if c in X.columns]

# 5) Sécurité : dropper tout 'object' restant non déclaré catégoriel
bad_obj = [c for c in X.columns if X[c].dtype == object and c not in cat_cols]
if bad_obj:
    print("Dropping non-categorical object columns:", bad_obj)
    X = X.drop(columns=bad_obj)

# Load the raw file
with open("../data/quoteai_last_50000.json", "r") as f:
    raw_quotes = json.load(f)

# Parsing quotes
accepted_quotes = [q for q in raw_quotes if q.get("proposed_price_accepted") is not None]
rows = []
for q in accepted_quotes:
    subprojects = q.get("subproject", [])
    for sub in subprojects:
        if all(k in sub for k in ["source_language", "target_language", "unit", "number_of_unit", "document_type"]):
            deadline_accepted = q.get("proposed_deadlines_accepted")
            if deadline_accepted is not None:
                rows.append({
                    "source_language": sub["source_language"],
                    "target_language": sub["target_language"],
                    "unit": sub["unit"],
                    "number_of_unit": sub["number_of_unit"],
                    "document_type": sub["document_type"],
                    "requested_deadline": deadline_accepted,
                    "proposed_price_accepted": q["proposed_price_accepted"]
                })

df = pd.DataFrame(rows)

# Handle missing deadline
df["requested_deadline"] = pd.to_numeric(df["requested_deadline"], errors="coerce")
median_deadline = df["requested_deadline"].median()
df["requested_deadline"] = df["requested_deadline"].fillna(median_deadline)

# Feature engineering
df["is_express"] = (df["requested_deadline"] <= 2).astype(int)
df["flexibility"] = df["requested_deadline"] - 1
df["has_rare_doc"] = df["document_type"].apply(lambda d: isinstance(d, str) and "contract" not in d.lower())
df["lang_pair"] = df["source_language"] + "_to_" + df["target_language"]

df["doc_category"] = df["document_type"].str.split("|").str[0]
df["doc_subcategory"] = df["document_type"].str.split("|").str[1]
df.drop(columns=["document_type"], inplace=True)

In [ ]:
# --- Removed global outliers (InterQuantileRange) ---
q1 = df["proposed_price_accepted"].quantile(0.25)
q3 = df["proposed_price_accepted"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df = df[(df["proposed_price_accepted"] >= lower_bound) & (df["proposed_price_accepted"] <= upper_bound)]

# --- Preparation for clustering ---
features_for_clustering = df.drop(columns=[
    "proposed_price_accepted",
    "source_language", "target_language",
    "doc_category", "doc_subcategory"
]).copy()

for col in features_for_clustering.columns:
    if features_for_clustering[col].dtype == "object":
        features_for_clustering[col] = features_for_clustering[col].astype("category").cat.codes

features_for_clustering = features_for_clustering.fillna(-999)

stds = features_for_clustering.std()
cols_with_variance = stds[stds > 0].index.tolist()
features_std = features_for_clustering[cols_with_variance]

features_std = (features_std - features_std.mean()) / features_std.std()

# --- Clustering + removal of outlier quotes by cluster ---
kmeans = KMeans(n_clusters=10, random_state=42, n_init="auto")
df["cluster"] = kmeans.fit_predict(features_std)

df["log_price"] = np.log1p(df["proposed_price_accepted"])
df["z_score"] = df.groupby("cluster")["log_price"].transform(zscore)

df = df[df["z_score"].abs() < 3]
df.drop(columns=["cluster", "log_price", "z_score"], inplace=True)

In [ ]:
# Define target and features
y = np.log1p(df["proposed_price_accepted"])
X = df.drop(columns=["proposed_price_accepted"])

X = X.fillna(-999)

# Identify categorical features for CatBoost
cat_cols = ["source_language", "target_language", "unit", "lang_pair", "doc_category", "doc_subcategory"]

In [ ]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 500),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]),
        "loss_function": "MAE",
        "verbose": 0,
        "random_seed": 42
    }

    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    maes = []

    for train_idx, valid_idx in cv.split(X):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        train_pool = Pool(X_train, y_train, cat_features=cat_cols)
        valid_pool = Pool(X_valid, y_valid, cat_features=cat_cols)

        model = CatBoostRegressor(**params)
        model.fit(train_pool)

        y_pred = model.predict(valid_pool)
        maes.append(mean_absolute_error(y_valid, y_pred))

    return np.mean(maes)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print("Best trial:")
print(study.best_trial.params)

In [ ]:
best_params = study.best_trial.params
best_params.update({
    "loss_function": "MAE",
    "verbose": 100,
    "random_seed": 42
})

train_pool = Pool(X, y, cat_features=cat_cols)
final_model = CatBoostRegressor(**best_params)
final_model.fit(train_pool)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostRegressor(**best_params)
model.fit(Pool(X_train, y_train, cat_features=cat_cols))

y_pred_log = model.predict(Pool(X_test, cat_features=cat_cols))
y_pred = np.expm1(y_pred_log)
y_test_exp = np.expm1(y_test)

mae = mean_absolute_error(y_test_exp, y_pred)
mape = np.mean(np.abs((y_test_exp - y_pred) / y_test_exp)) * 100
smape = 100 * np.mean(2 * np.abs(y_pred - y_test_exp) / (np.abs(y_test_exp) + np.abs(y_pred)))
mae_weighted = np.average(np.abs(y_test - y_pred_log), weights=y_test)

print(f"MAE: {mae:.2f} €")
print(f"MAPE: {mape:.2f} %")
print(f"SMAPE: {smape:.2f} %")
print(f"Weighted MAE (log-scale): {mae_weighted:.4f}")

In [ ]:
# --- SHAP for CatBoost (compatible anciennes/nouvelles versions) ---
import shap, matplotlib.pyplot as plt, inspect
from catboost import Pool
import pandas as pd
import numpy as np

def _supports_kw(func, kw: str) -> bool:
    try:
        return kw in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False

# 1) Sous-échantillon
N = min(5000, len(X_test))
X_sample = X_test.sample(N, random_state=42).copy()

# 2) ShapValues depuis CatBoost (dernière colonne = expected value)
pool_sample = Pool(X_sample, cat_features=[c for c in cat_cols if c in X_sample.columns])
shap_arr = model.get_feature_importance(pool_sample, type="ShapValues")
shap_values = shap_arr[:, :-1]
base_value  = shap_arr[:, -1].mean()

# 3) Encodage numérique pour les colonnes catégorielles (pour le tri)
X_num = X_sample.copy()
for c in X_num.columns:
    if str(X_num[c].dtype) == "category" or X_num[c].dtype == object:
        X_num[c] = pd.Categorical(X_num[c]).codes

# 4) Summary plots (fallback si 'display_features' non supporté)
if _supports_kw(shap.summary_plot, "display_features"):
    shap.summary_plot(shap_values, X_num,
                      feature_names=list(X_sample.columns),
                      display_features=X_sample, show=False)
else:
    shap.summary_plot(shap_values, X_num,
                      feature_names=list(X_sample.columns),
                      show=False)
plt.tight_layout(); plt.savefig("shap_summary_cat.png", dpi=200); plt.close()

if _supports_kw(shap.summary_plot, "display_features"):
    shap.summary_plot(shap_values, X_num,
                      feature_names=list(X_sample.columns),
                      display_features=X_sample, plot_type="bar", show=False)
else:
    shap.summary_plot(shap_values, X_num,
                      feature_names=list(X_sample.columns),
                      plot_type="bar", show=False)
plt.tight_layout(); plt.savefig("shap_bar_cat.png", dpi=200); plt.close()

# 5) Dependence plots (même fallback)
def safe_dependence(feat):
    if feat in X_num.columns:
        if _supports_kw(shap.dependence_plot, "display_features"):
            shap.dependence_plot(feat, shap_values, X_num,
                                 display_features=X_sample, show=False)
        else:
            shap.dependence_plot(feat, shap_values, X_num, show=False)
        plt.tight_layout(); plt.savefig(f"shap_dependence_cat_{feat}.png", dpi=200); plt.close()

for feat in ["number_of_unit", "requested_deadline", "doc_category", "lang_pair"]:
    safe_dependence(feat)

In [ ]:
# --- Metrics by segment (doc_type category, language pair, deadline bins) ---
import numpy as np
import pandas as pd

# Reconstruire un DF de test exploitable
test_df = X_test.copy()
test_df["y_true"] = np.expm1(y_test)
test_df["y_pred"] = np.expm1(y_pred_log)  # y_pred_log vient de ta cellule d'évaluation

def seg_metrics(df, by, bins=None):
    if bins is not None:
        s = pd.cut(df[by], bins=bins, include_lowest=True)
    else:
        s = df[by]
    g = df.groupby(s)
    out = g.apply(lambda d: pd.Series({
        "N": len(d),
        "MAE (€)": np.mean(np.abs(d["y_true"] - d["y_pred"])),
        "MAPE (%)": 100*np.mean(np.abs(d["y_true"]-d["y_pred"])/np.clip(d["y_true"], 1e-9, None)),
        "SMAPE (%)": 100*np.mean(2*np.abs(d["y_pred"]-d["y_true"])/(np.abs(d["y_true"])+np.abs(d["y_pred"]))),
    }))
    return out.sort_values("N", ascending=False)

# Doc category
m_doc = seg_metrics(
    test_df.assign(doc_category=X_test["doc_category"].astype(str)),
    "doc_category"
)
m_doc.to_csv("metrics_by_doc_cat.csv")
m_doc.head(20)

# Lang pair (si présent dans X)
if "lang_pair" in X_test.columns:
    m_lang = seg_metrics(test_df.assign(lang_pair=X_test["lang_pair"].astype(str)), "lang_pair")
    m_lang.to_csv("metrics_by_lang_pair.csv")
    m_lang.head(20)

# Délai : binnings simples
m_deadline = seg_metrics(test_df, "requested_deadline", bins=[-0.1, 1, 2, 3, 5, 10, 30, 1000])
m_deadline.to_csv("metrics_by_deadline_bins.csv")
m_deadline

In [ ]:
# --- Ablations rapides (retrain léger sur le même split) ---
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error

def train_eval_catboost(Xtr, ytr, Xte, yte, cat_cols, params):
    m = CatBoostRegressor(**params)
    m.fit(Pool(Xtr, ytr, cat_features=cat_cols), verbose=0)
    yp = m.predict(Pool(Xte, cat_features=cat_cols))
    return {
        "MAE (€)": mean_absolute_error(np.expm1(yte), np.expm1(yp)),
        "MAPE (%)": 100*np.mean(np.abs(np.expm1(yte)-np.expm1(yp))/np.clip(np.expm1(yte),1e-9,None))
    }

base_params = {**best_params, "loss_function": "MAE", "verbose": 0}

# A1) Sans la famille "deadline"
drop_deadline = ["requested_deadline","is_express","flexibility","urgency","delay_interaction"]
cols_keep = [c for c in X_train.columns if c not in drop_deadline]
res_no_deadline = train_eval_catboost(X_train[cols_keep], y_train, X_test[cols_keep], y_test, [c for c in cat_cols if c in cols_keep], base_params)

# A2) Sans outliers (déjà ton pipeline les enlève). Pour simuler l'impact,
#     on compare "baseline" vs "retrain" sur le même split mais en réinjectant
#     grossièrement des points extrêmes (si tu as df_raw sous la main, refais un split depuis df_raw).
res_baseline = train_eval_catboost(X_train, y_train, X_test, y_test, cat_cols, base_params)

pd.DataFrame([res_baseline, res_no_deadline], index=["Baseline", "No deadline"])

In [ ]:
comparison_df = pd.DataFrame({
    "Actual price (€)": y_test_exp,
    "Predicted price (€)": y_pred
})
comparison_df["Within 5€?"] = (comparison_df["Actual price (€)"] - comparison_df["Predicted price (€)"]).abs() <= 5

print(comparison_df.head(30))